# DeepFaceLive — Smooth GPU (Google Colab) 🚀

Run the face-swap app on a **free Colab GPU** so the celebrity (DFM) deepfake is **smooth**, not laggy.

## Just 2 steps
1. **Turn the GPU on:** top menu → **Runtime → Change runtime type → T4 GPU → Save**.
2. **Tap the ▶️ button** on the cell below and wait about 2 minutes. A **QR code + link** appear — **scan the QR with your phone camera** (or type the link) to open the app on your phone.

Then on your phone: pick a celebrity model → **Load Model** → **Start Camera**. It stays smooth because you are on the GPU.

> Keep this Colab tab open while you use the app. When you close it the link stops working — that is normal; just reopen this page and tap ▶️ again for a fresh link.
>
> Bookmark this page: `https://colab.research.google.com/github/oluwacoded/Deepfaketrial/blob/colab-gpu/deepfacelive_colab.ipynb`


In [ ]:
#@title ▶️  Tap this button, then wait ~2 minutes for your phone link { display-mode: "form" }
# One tap does everything: download the app, install the GPU runtime, start the
# server, and show a QR + link for your phone.
# FIRST turn the GPU on:  Runtime -> Change runtime type -> T4 GPU -> Save.

import os, sys, re, time, glob, site, subprocess, threading
import urllib.request, urllib.error

def sh(cmd):
    return subprocess.run(cmd, shell=True, capture_output=True, text=True)

# 0) Is a GPU actually switched on?
gpu = sh("nvidia-smi -L")
if "GPU" not in gpu.stdout:
    print("=" * 64)
    print("  NO GPU IS ON.")
    print("  Do this:  Runtime  ->  Change runtime type  ->  T4 GPU  ->  Save")
    print("  Then tap the play button on this cell again.")
    print("=" * 64)
    raise SystemExit

print("GPU detected:", gpu.stdout.strip().split(":", 1)[-1].strip())

# 1) Download a fresh copy of the app
os.chdir("/content")
sh("rm -rf Deepfaketrial")
print("Downloading the app...")
r = sh("git clone --depth 1 -b colab-gpu https://github.com/oluwacoded/Deepfaketrial.git")
if r.returncode != 0:
    print(r.stdout, r.stderr)
    raise SystemExit("Download failed - tap the play button again.")
os.chdir("/content/Deepfaketrial")

# 2) Install the GPU runtime. The [cuda,cudnn] extra makes onnxruntime bring its
#    OWN matching CUDA + cuDNN, which fixes the "libcudnn.so.9 not found" crash.
print("Installing the GPU runtime (about 2 minutes)...")
r = sh(sys.executable + ' -m pip install -q "onnxruntime-gpu[cuda,cudnn]" '
       'flask flask-socketio eventlet numexpr h5py onnx qrcode')
if r.returncode != 0:
    print(r.stdout[-2000:], r.stderr[-2000:])
    raise SystemExit("Install failed - tap the play button again.")

# 3) Put the freshly installed CUDA/cuDNN libraries on the path for the server
#    process (works together with the app's own preload_dlls()).
lib_dirs = []
for sp in set(site.getsitepackages() + [site.getusersitepackages()]):
    lib_dirs += glob.glob(os.path.join(sp, "nvidia", "*", "lib"))
env = dict(os.environ)
env["LD_LIBRARY_PATH"] = ":".join(lib_dirs) + ":" + env.get("LD_LIBRARY_PATH", "")

# 4) cloudflared gives a public https link your phone can open
sh("wget -q https://github.com/cloudflare/cloudflared/releases/latest/download/cloudflared-linux-amd64 -O /usr/local/bin/cloudflared")
sh("chmod +x /usr/local/bin/cloudflared")

# 5) Start the app
def drain(p, tag):
    for line in p.stdout:
        print(tag, line, end="")

print("Starting the app...")
server = subprocess.Popen([sys.executable, "web_server.py"],
    stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True, bufsize=1, env=env)
threading.Thread(target=drain, args=(server, "[app]"), daemon=True).start()

ready = False
deadline = time.time() + 150
while time.time() < deadline:
    if server.poll() is not None:
        break
    try:
        urllib.request.urlopen("http://localhost:5000/", timeout=2); ready = True; break
    except urllib.error.HTTPError:
        ready = True; break
    except Exception:
        time.sleep(1)

if not ready:
    print("=" * 64)
    print("  The app did not start. Read the [app] lines above - the last one")
    print("  says why. Usually: tap the play button again, or reconnect the GPU.")
    print("=" * 64)
else:
    tunnel = subprocess.Popen(
        ["cloudflared", "tunnel", "--url", "http://localhost:5000", "--no-autoupdate"],
        stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True, bufsize=1)
    url = None
    deadline = time.time() + 40
    while time.time() < deadline:
        line = tunnel.stdout.readline()
        if not line:
            break
        m = re.search(r"https://[-a-z0-9]+\.trycloudflare\.com", line)
        if m:
            url = m.group(0); break
    threading.Thread(target=drain, args=(tunnel, "[link]"), daemon=True).start()
    print("=" * 64)
    if url:
        print("  OPEN THIS ON YOUR PHONE:")
        print("        " + url)
        print("=" * 64)
        try:
            import qrcode
            from IPython.display import display
            print("  ...or scan this QR code with your phone camera:")
            display(qrcode.make(url))
        except Exception as e:
            print("  (QR unavailable - just type the link above.)", e)
    else:
        print("  Could not read the link automatically.")
        print("  Look in the [link] lines above for a .trycloudflare.com address.")
    print("=" * 64)
    print("  Keep this Colab tab OPEN while you use the app.")
